"Revenue Growth through Funnel Optimization and Campaign Efficiency"


Our e-commerce store is generating traffic, but the management is concerned about two things:

High Leakage: Many users visit, but very few actually complete a purchase.

Marketing Waste: We are spending money on campaigns, but we don't know which ones are actually bringing in "new" high-value customers versus just giving discounts to people who would have bought anyway.

Objective 1: Customer Value Segmentation (The "Who")
- Goal: Identify which customer groups are actually making the company money.
- KPIs: Average Order Value (AOV) by loyalty_tier and country.
- Action: Join Customers + Transactions.

Objective 2: Conversion Funnel Analysis (The "Where")
- Goal: Find the exact page or step where we lose the most money.
- KPIs: Drop-off rate from view → add_to_cart → purchase.
- Action: Analyze the events table to see where the "path to purchase" breaks.

Objective 3: Marketing Campaign ROI (The "How")
- Goal: Determine if our "Promotion" budget is being spent effectively.
- KPIs: Revenue per Campaign vs. Expected Uplift.
- Action: Join Campaigns + Transactions.

Objective 4: Return/Refund Investigation (The "Risk")
- Goal: Reduce losses from dissatisfied customers.
- KPIs: Refund Rate per Product Category and Discount Level.
- Action: Use the refund_flag in the transactions table.

## Database setup

In [2]:
import pandas as pd
import numpy as np
import os
import sqlalchemy as sa
import urllib
from dotenv import load_dotenv
import logging
import time

In [ ]:
for file in os.listdir('data'):
    if '.csv' in file:
        print(file)

In [ ]:
for file in os.listdir('data'):
    if '.csv' in file:
        df=pd.read_csv('data/'+file,on_bad_lines='skip')
        print(f"{file} : {df.shape}")
        print(df.columns)
        
        # Used when the file fie is in parquet format
    #elif '.parquet' in file:
        #df=pd.read_parquet('data/'+file,engine='fastparquet')
        #print(f"{file} : {df.shape}")
        #print(df.columns)

In [ ]:
{
            'customers':'customer_id',
            'orders':'order_id',
            'products':'product_id',
            'sellers':'seller_id',
            'order_reviews':'review_id',
            'geolocation': 'geolocation_zip_code_prefix',
            'order_items': 'order_id',
            'order_payments': 'order_id'

        }

### assumption to be working on SQL SERVER

In [ ]:
#function to create a engine
    # call load_dotenv
        #import credentials to connect to sql_server

    # create a connection string,this will also be used when the the database is already present just the replacing database type and database name
    # get params from the connection string
    # create engine to connect to master db when the db is not present
    # connect and create a database with new_db_name
#### when the db is already present
    #map/store primary_key of each table in dictionary
    #convert master db connction string  to main db connection string
    #get final params of main db connection string
    #create main db engine


# function to load raw data or update the existing tables
    #get csv filenames from local machine
    # loop
        #read the files
        #call update_insert function to update/insert to db

# function to update_insert to db
    #inspect the engine
    #check if the table exist then using staging table update or create a new table
    #create stg_table with table name if exist
    #add/replace staging table in db
    #merge statements
        #get column name except the primary key
        #get set of columns to update
        #get columns to insert
        #get values to insert into columns
    #execute the merge and drop staging table


### connecting/creating database

In [ ]:
# connecting to database
server_name='localhost'
user_name='sa'
password='KumarSushant@20081996'
new_db_name='marketing_e_commerce'

# creating connection string to connect to master db
master_db_connection_string=(
    f'DRIVER={{ODBC Driver 17 for SQL SERVER}};'
    f'SERVER={{{server_name}}};'
    f'DATABASE=master;'
    f'UID={{{user_name}}};'
    f'PWD={{{password}}};'
    f'Encrypt=yes;'
    f'TrustServerCertificate=yes;'
)

#get params from the connection string
master_db_params=urllib.parse.quote_plus(master_db_connection_string)
print(master_db_params)

# create master db engine
master_db_engine=sa.create_engine(url=f'mssql+pyodbc:///?odbc_connect={master_db_params}',isolation_level='AUTOCOMMIT')
print(master_db_engine)

In [ ]:
#opening(with) and connecting to master db when the main db is not present
with master_db_engine.connect() as conn:
    # if the databse exist then make ready for data ingestion else create and make ready for ingestion
    conn.execute(sa.text(f"if not exists (select * from sys.databases where name='{new_db_name}') create database {new_db_name};"))
    print(f"Database {new_db_name}Created, Ready for Ingestion")

In [ ]:
#when the main db is present

#create a primary_key map
pk_map={
    'listings':'listing_id',
    'past_rates':'listing_id'
    
}

pkmap=pk_map
# converting master_db_connection_string to main_db_connection_string
main_db_connection_string=master_db_connection_string.replace("DATABASE=master;",f"DATABASE={new_db_name}")

main_db_params=urllib.parse.quote_plus(main_db_connection_string)

main_db_engine=sa.create_engine(f"mssql+pyodbc:///?odbc_connect={main_db_params}",isolation_level='AUTOCOMMIT')

### collating into a function

In [7]:
#create a primary_key map


def connect_to_db():
    load_dotenv()
    # connecting to database
    server_name=os.getenv('db_server')
    user_name=os.getenv('db_username')
    password=os.getenv('db_password')
    new_db_name='marketing_e_commerce'

    # creating connection string to connect to master db
    master_db_connection_string=(
        f'DRIVER={{ODBC Driver 17 for SQL SERVER}};'
        f'SERVER={{{server_name}}};'
        f'DATABASE=master;'
        f'UID={{{user_name}}};'
        f'PWD={{{password}}};'
        f'Encrypt=yes;'
        f'TrustServerCertificate=yes;'
    )

    #get params from the connection string
    master_db_params=urllib.parse.quote_plus(master_db_connection_string)

    # create master db engine
    master_db_engine=sa.create_engine(url=f'mssql+pyodbc:///?odbc_connect={master_db_params}',isolation_level='AUTOCOMMIT')

    #opening(with) and connecting to master db when the main db is not present
    with master_db_engine.connect() as conn:
        # if the databse exist then make ready for data ingestion else create and make ready for ingestion
        conn.execute(sa.text(f"if not exists (select * from sys.databases where name='{new_db_name}') create database {new_db_name};"))
        print(f"Database {new_db_name} Created, Ready for Ingestion")


    # converting master_db_connection_string to main_db_connection_string when the main db is present
    main_db_connection_string=master_db_connection_string.replace("DATABASE=master;",f"DATABASE={new_db_name};")

    main_db_params=urllib.parse.quote_plus(main_db_connection_string)

    main_db_engine=sa.create_engine(url=f'mssql+pyodbc:///?odbc_connect={main_db_params}',isolation_level='AUTOCOMMIT')

    return main_db_engine



In [9]:
    #when the main db is present
pk_map={
    'campaigns':'campaign_id',
    'customers':'customer_id',
    'events':'event_id',
    'products':'product_id',
    'transactions':'transaction_id'
    
}


pkmap=pk_map

data_files=[file for file in os.listdir('data')]
engine=connect_to_db()
for file in data_files:
    if '.csv' in file:
        df=pd.read_csv(f'data/'+file)
        table_name=file[0:-5]
        pk=pkmap.get(table_name,'id')
        #upsert_to_db(df=df,table_name=table_name,engine=engine,primary_key=pk)

C:\Users\suman\AppData\Local\Temp\ipykernel_19280\2366600644.py:30: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with master_db_engine.connect() as conn:


Database marketing_e_commerce Created, Ready for Ingestion


In [ ]:
transactional_tables=['transactions']
static_tables=['campaigns','customers','events','products','master']

def upsert_to_db(df,table_name,engine,primary_key):
    # Creating Table if not in DB
    inspector=sa.inspect(engine)
    primary_key=pk
    if not inspector.has_table(table_name=table_name,schema='dbo'):
        print(f'Table {table_name} Does not Exist Creating It....')
        df.to_sql(name=table_name,con=engine,schema='dbo',if_exists='replace',index=False)
        print(f"Table {table_name} created successfully")
        
    #Creating staging table
    staging_table=f'stg_{table_name}'
    df.to_sql(name=staging_table,con=engine,if_exists='replace',schema='dbo',index=False)


    #merging
    columns_to_update=[col for col in df.columns if col!=primary_key]
    update_strings=",".join([f"target.[{col}]=source.[{col}]" for col in columns_to_update])
    insert_cols=",".join(f"[{col}]" for col in df.columns)
    insert_values=",".join([f"source.[{col}]" for col in df.columns])

    upsert_query = f"""
            MERGE INTO dbo.[{table_name}] AS target
            USING dbo.[{staging_table}] AS source
            ON (target.[{primary_key}] = source.[{primary_key}])
            WHEN MATCHED THEN
                UPDATE SET {update_strings}
            WHEN NOT MATCHED THEN
                INSERT ({insert_cols})
                VALUES ({insert_values});
            """

    append_query=f"""
    INSERT INTO dbo.[{table_name}] ({insert_cols})
    SELECT {insert_cols}
    FROM dbo.[{staging_table}];
    """


    if table_name in static_tables:
        with engine.begin() as conn:
            conn.execute(sa.text(upsert_query))
            conn.execute(sa.text(f"drop table dbo.{staging_table}"))
        print(f"Upsert for {table_name} completed successfully.")
        
    else:
        with engine.begin() as conn:
            conn.execute(sa.text(append_query))
            conn.execute(sa.text(f"drop table dbo.{staging_table}"))
        print(f"Append for {table_name} completed successfully.")

In [ ]:
pk_map={
    'campaigns':'campaign_id',
    'customers':'customer_id',
    'events':'event_id',
    'products':'product_id',
    'transactions':'transaction_id'
    
}

def ingest_data():
    
    pkmap=pk_map

    data_files=[file for file in os.listdir('data')]
    engine=connect_to_db()
    for file in data_files:
        if '.csv' in file:
            df=pd.read_csv(f'data/'+file)
            table_name=file[0:-5]
            pk=pkmap.get(table_name,'id')
            upsert_to_db(df=df,table_name=table_name,engine=engine,primary_key=pk)

In [1]:
rag=[]
len(rag)

0

In [26]:
data_files=[file for file in os.listdir('initial_data')]
pk_map={}
for file in data_files:
    if '.csv' in file:
        df=pd.read_csv(f'initial_data/{file}')
        print(df.shape)
        table_name=file[:-4]
        df_headers=pd.read_csv(f"initial_data/{file}",nrows=0)
        lower_headers=[col.lower() for col in df_headers.columns]
        expected_pk=f"{table_name[:-1]}_id".lower()

        if expected_pk in lower_headers:
            original_col_name=df_headers.columns[lower_headers.index(expected_pk)]
            pk_map[table_name]=original_col_name
print(pk_map)

(50, 7)
(100000, 7)
(2000000, 12)
(2000, 6)
(103127, 9)
{'campaigns': 'campaign_id', 'customers': 'customer_id', 'events': 'event_id', 'products': 'product_id', 'transactions': 'transaction_id'}


In [24]:
def get_primary_keys(data_folder_name:str):
    data_files=[file for file in os.listdir(data_folder_name)]
    pk_map={}
    for file in data_files:
        if '.csv' in file:
            table_name=file[:-5]
            df_headers=pd.read_csv(f"initial_data/{file}",nrows=0)
            lower_headers=[col.lower() for col in df_headers.columns]
            expected_pk=f"{table_name}_id".lower()

            if expected_pk in lower_headers:
                original_col_name=df_headers.columns[lower_headers.index(expected_pk)]
                pk_map[table_name]=original_col_name
    return pk_map

In [25]:
get_primary_keys('initial_data')

{'campaign': 'campaign_id',
 'customer': 'customer_id',
 'event': 'event_id',
 'product': 'product_id',
 'transaction': 'transaction_id'}